# 07 - Rerank candidates with AlignScore

The source abstract is the context. One summary is selected independently within each penalty's group of five; the first maximum breaks exact ties. Candidate-level scores are retained.

## Fixed Configuration

In [1]:
from pathlib import Path
import json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
shared_notebook = ROOT / 'notebooks/05_prepare_validation_ablation.ipynb'
shared_cells = json.loads(shared_notebook.read_text(encoding='utf-8'))['cells']
for shared_index, shared_cell in enumerate(shared_cells):
    if shared_cell['cell_type'] == 'code' and 'ablation-shared' in shared_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(shared_cell['source']), str(shared_notebook) + ':cell' + str(shared_index), 'exec'), globals())
ablation = SummarizationAblation(CONFIG)
ablation.validate_run()


## Reranking implementation
AlignScore uses the pinned metric snapshot and checkpoint. Human references and SummaC are not used for candidate selection. Progress saves per abstract.


In [2]:
def rerank_candidates(self):
    self.validate_run()
    self.configure_metric_environment()
    import torch
    import transformers
    from tqdm.auto import tqdm
    transformers.AdamW = torch.optim.AdamW
    from alignscore import AlignScore
    ckpt = ROOT / self.c['alignscore']['checkpoint']
    require(ckpt.is_file(), f'Missing AlignScore checkpoint: {ckpt}')
    # Allow full loading of the Lightning checkpoint.
    os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
    a = self.c['alignscore']
    scorer = AlignScore(model=a['model'], batch_size=a['batch_size'], device=self.c['device'],
                        ckpt_path=str(ckpt), evaluation_mode=a['evaluation_mode'], verbose=False)
    for penalty in self.c['penalties']:
        src = self.out / f'diverse_beam/penalty_{penalty}.csv'
        candidates = self.load_completed_output(src)
        get_stage_manifest = self.get_stage_manifest(f'align_{penalty}', [src, ckpt])
        output = self.out / f'reranked/penalty_{penalty}.csv'
        rows = self.load_progress(output, self.data, get_stage_manifest)
        for i in tqdm(range(len(rows), len(self.data)), desc=f'AlignScore {penalty}'):
            texts = [candidates[i][f'candidate_{j}'] for j in range(1, 6)]
            with torch.inference_mode():
                scores = [float(x) for x in scorer.score(contexts=[self.data[i]['abstract']]*5, claims=texts)]
            valid_scores(scores, 5)
            best = max(range(5), key=lambda j: scores[j])
            rows.append(dict(id=self.ids[i], selected_summary=texts[best], selected_alignscore=scores[best],
                             selected_candidate=best+1, **{f'alignscore_{j+1}': s for j, s in enumerate(scores)}))
            self.save_progress(output, rows, get_stage_manifest)
    inspect_inputs = []
    for p in self.c['penalties']:
        candidates = self.load_completed_output(self.out / f'diverse_beam/penalty_{p}.csv')
        selected = self.load_completed_output(self.out / f'reranked/penalty_{p}.csv')
        for c, r in zip(candidates, selected):
            for j in range(1, 6):
                inspect_inputs.append(dict(id=r['id'], penalty=p, candidate=j, summary=c[f'candidate_{j}'], alignscore=r[f'alignscore_{j}']))
    save_csv(self.out / 'scores/candidate_alignscores.csv', inspect_inputs)


SummarizationAblation.rerank_candidates = rerank_candidates


## AlignScore reranking
Outputs: three reranked CSVs plus scores/candidate_alignscores.csv.


In [3]:
ablation.rerank_candidates()


c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv-scoring\Lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv-scoring\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:51: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, 

AlignScore 0.5:   0%|          | 0/619 [00:00<?, ?it/s]

AlignScore 1.0:   0%|          | 0/619 [00:00<?, ?it/s]

AlignScore 2.0:   0%|          | 0/619 [00:00<?, ?it/s]

## Selection coverage
Expected total: 1,857 selected summaries, with five candidate scores per abstract and penalty.


In [4]:
selected_total = 0
for penalty in CONFIG['penalties']:
    rows = ablation.load_completed_output(ablation.out / f'reranked/penalty_{penalty}.csv')
    print(f'Penalty {penalty}: {len(rows)} selected summaries')
    selected_total += len(rows)
require(selected_total == 1857, 'Selection coverage is incomplete')


Penalty 0.5: 619 selected summaries
Penalty 1.0: 619 selected summaries
Penalty 2.0: 619 selected summaries
